# Lab 02 — Chapter 13: Experiment 1 — Corpus statistics

Notebook này thống kê corpus C4 trước khi xây dựng language model. Chạy các cell theo thứ tự.

## 1. Import và cấu hình

In [ ]:
from collections import Counter
from pathlib import Path
from typing import Sequence
import json
import re

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'lab2').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'lab2').is_dir():
    raise FileNotFoundError('Không tìm thấy thư mục lab2.')

DATASET_PATH = PROJECT_ROOT / 'data' / 'c4-train.00000-of-01024-30K.json' / 'c4-train.00000-of-01024-30K.json'
DOCUMENT_LIMIT = 10_000
NGRAM_ORDERS = (1, 2, 3)
MODEL_NAMES = {1: 'unigram', 2: 'bigram', 3: 'trigram'}
TOKEN_PATTERN = re.compile(r"[a-z]+(?:'[a-z]+)?")
SENTENCE_BOUNDARY_PATTERN = re.compile(r'[.!?]+(?:\s+|$)')


## 2. Tải và tokenize corpus

In [ ]:
def tokenize(text: str) -> list[str]:
    """Chuẩn hóa một đoạn văn bản thành danh sách token chữ thường."""
    return TOKEN_PATTERN.findall(text.lower())


def load_sentences(dataset_path: Path, document_limit: int) -> tuple[int, list[list[str]]]:
    """Đọc JSONL, lấy tối đa số documents yêu cầu và tách thành các câu có token."""
    document_count = 0
    sentences: list[list[str]] = []
    with dataset_path.open(encoding='utf-8') as dataset_file:
        for line in dataset_file:
            document = json.loads(line).get('text', '')
            if not isinstance(document, str) or not document.strip():
                continue
            sentence_texts = SENTENCE_BOUNDARY_PATTERN.split(document)
            sentences.extend(tokens for tokens in map(tokenize, sentence_texts) if tokens)
            document_count += 1
            if document_count >= document_limit:
                break
    if not sentences:
        raise ValueError('Corpus không có câu hợp lệ.')
    return document_count, sentences


document_count, sentences = load_sentences(DATASET_PATH, DOCUMENT_LIMIT)
print(f'Documents: {document_count:,}')
print(f'Sentences: {len(sentences):,}')


## 3. Đếm unigram, bigram và trigram

In [ ]:
def count_ngrams(corpus: Sequence[Sequence[str]], order: int) -> Counter[tuple[str, ...]]:
    """Đếm n-gram liên tiếp theo order trong toàn bộ corpus đã tokenize."""
    ngram_counts: Counter[tuple[str, ...]] = Counter()
    for sentence in corpus:
        for start_index in range(max(0, len(sentence) - order + 1)):
            ngram_counts[tuple(sentence[start_index:start_index + order])] += 1
    return ngram_counts


token_count = sum(len(sentence) for sentence in sentences)
vocabulary = {token for sentence in sentences for token in sentence}
corpus_statistics = {
    'documents': document_count,
    'sentences': len(sentences),
    'tokens': token_count,
    'vocabulary_size': len(vocabulary),
}
ngram_counts_by_order = {order: count_ngrams(sentences, order) for order in NGRAM_ORDERS}
ngram_statistics = [
    {
        'model': MODEL_NAMES[order],
        'unique_ngrams': len(ngram_counts_by_order[order]),
        'hapax_ngrams': sum(count == 1 for count in ngram_counts_by_order[order].values()),
    }
    for order in NGRAM_ORDERS
]
display(corpus_statistics)
display(ngram_statistics)


## 4. Vẽ frequency distribution của n-gram

In [ ]:
import matplotlib.pyplot as plt

frequency_distributions = {
    order: Counter(ngram_counts.values())
    for order, ngram_counts in ngram_counts_by_order.items()
}
figure, axes = plt.subplots(nrows=1, ncols=len(NGRAM_ORDERS), figsize=(15, 4))
for axis, order in zip(axes, NGRAM_ORDERS):
    distribution = frequency_distributions[order]
    axis.scatter(list(distribution), list(distribution.values()), s=10)
    axis.set_xscale('log')
    axis.set_yscale('log')
    axis.set_title(MODEL_NAMES[order])
    axis.set_xlabel('Frequency')
    axis.set_ylabel('Number of n-gram types')
figure.suptitle('Frequency distribution of observed n-grams')
figure.tight_layout()
plt.show()
